# Video indexing at speed: Kaggle development and measurement log

This notebook is the working log behind `run.py` ([repo](https://github.com/mahamasiff/alethia-assignment)).
It shows how the accuracy settings were chosen and how the timings in the README were measured.

- **Hardware:** Kaggle, 2 × Tesla T4, 4 vCPUs. This is not the grading machine (RTX 4090). See `HARDWARE.md`.
- **Timed runs** use the grader's exact command, `python run.py --input … --output …`, with **default settings: FP16, 5 frames analysed per second of video**.
- **Accuracy** was measured against an FP32, every-frame reference built here on a sample (section 4).
- **Results, coverage and the speed table** are in the README.

Run order: sections 0–14 were the development runs, in the order they happened. Sections A–D are the batch run over the remaining parquet files.

## 0. Clone the repo and install dependencies (first setup)

Kaggle's own Python 3.13 and torch 2.11 (CUDA 13) are used as they are, so this installs only `ultralytics` and `av`.
`requirements.txt` pins versions for the grader's Python 3.11 / CUDA 12 instead. It is tested separately in sections 7–8.

In [4]:
!git clone https://github.com/mahamasiff/alethia-assignment.git
%cd alethia-assignment
!pip install -q ultralytics "av>=14" huggingface_hub

Cloning into 'alethia-assignment'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 1), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 11.02 KiB | 11.02 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/kaggle/working/alethia-assignment/alethia-assignment


## 1. Smoke test: 4 videos, first 20 s each, output format check

Clones or updates the repo, fetches the weights with `prepare.sh`, downloads `train-00000`, and runs `run.py` from a *different* folder to prove the weights are found next to `run.py`.
It then checks every line's keys, object names, people count, and that each video's seconds run exactly 0..max.

**Result:** exit 0 in 41 s; 4 videos × 20 seconds; no errors; `format OK`.

**Note:** this cell downloaded to `/kaggle/working/fv`. Every later section uses a copy of the data in **`/tmp/fv/data`** (files `train-00000` and `train-00001`), kept outside the saved output folder.

In [9]:
import os, json, time, subprocess
from collections import defaultdict

REPO = "/kaggle/working/alethia-assignment"
DATA = "/kaggle/working/fv"

# 1) code: clone or update
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/mahamasiff/alethia-assignment.git {REPO}
!git -C {REPO} log --oneline -1

# 2) deps for Kaggle's Python (requirements.txt targets the grader) + weights
!pip install -q ultralytics "av>=14"
!bash {REPO}/prepare.sh

# 3) one parquet file (~0.5 GB)
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
hf_hub_download("HuggingFaceFV/finevideo", "data/train-00000-of-01357.parquet", repo_type="dataset",
                local_dir=DATA, token=UserSecretsClient().get_secret("HF_TOKEN"))

# 4) smoke run: 4 videos, first 20 s each, both GPUs. Run from another folder on purpose,
#    to prove the weights are found next to run.py, not in the current directory.
os.chdir("/kaggle/working")
t0 = time.time()
r = subprocess.run(["python", f"{REPO}/run.py", "--input", f"{DATA}/data", "--output", "smoke.jsonl",
                    "--max-videos", "4", "--max-seconds", "20"], capture_output=True, text=True)
print(r.stderr[-3000:])
print(f"exit code {r.returncode}, {time.time() - t0:.0f}s")

# 5) check the output format
NAMES = {"person", "car", "dog", "laptop", "cell phone"}
secs, errs, problems = defaultdict(list), {}, []
for i, line in enumerate(open("smoke.jsonl")):
    d = json.loads(line)
    if "error" in d:
        errs[d["video"]] = d["error"]
        continue
    if set(d) != {"video", "second", "objects", "people"}:
        problems.append(f"line {i}: keys {set(d)}")
    if not set(d["objects"]) <= NAMES:
        problems.append(f"line {i}: bad objects {d['objects']}")
    if not (isinstance(d["people"], int) and d["people"] >= 0):
        problems.append(f"line {i}: bad people {d['people']}")
    secs[d["video"]].append(d["second"])
for v, s in secs.items():
    if sorted(s) != list(range(max(s) + 1)):
        problems.append(f"{v}: seconds not exactly 0..{max(s)}")
    print(f"{v}: {len(s)} seconds")
print("errors:", errs or "none")
print("PROBLEMS:\n" + "\n".join(problems) if problems else "format OK")
!head -5 smoke.jsonl

814527a (HEAD -> main, origin/main, origin/HEAD) Add prepare.sh to fetch yolo11l.pt; run.py loads weights offline from its own folder
prepare: /kaggle/working/alethia-assignment/yolo11l.pt already present, skipping download
prepare: /kaggle/working/alethia-assignment/yolo11l.pt ok (51387343 bytes)
prepare: ultralytics 8.4.216 loaded /kaggle/working/alethia-assignment/yolo11l.pt (80 classes)
prepare: done


data/train-00000-of-01357.parquet: reconstructing file:   0%|          |  0.00B /  494MB            

data/train-00000-of-01357.parquet: downloading bytes:           |  0.00B            

[16:26:56] 1 row groups, 4 videos, 2 worker(s) x 2 decoders, samples/sec=all, half=False
[16:27:16] [gpu1] videos 1  frames 600  34.6 fps  17s
[16:27:23] [gpu0] videos 1  frames 980  40.5 fps  24s
[16:27:23] [gpu0] videos 2  frames 980  40.5 fps  24s
[16:27:34] [gpu0] videos 3  frames 1576  44.5 fps  35s
[16:27:35] done in 38.5s -> smoke.jsonl

exit code 0, 41s
train-00000-of-01357.parquet:0: 20 seconds
train-00000-of-01357.parquet:2: 20 seconds
train-00000-of-01357.parquet:3: 20 seconds
train-00000-of-01357.parquet:1: 20 seconds
errors: none
format OK
{"video": "train-00000-of-01357.parquet:0", "second": 0, "objects": [], "people": 0}
{"video": "train-00000-of-01357.parquet:0", "second": 1, "objects": [], "people": 0}
{"video": "train-00000-of-01357.parquet:0", "second": 2, "objects": [], "people": 0}
{"video": "train-00000-of-01357.parquet:0", "second": 3, "objects": [], "people": 0}
{"video": "train-00000-of-01357.parquet:0", "second": 4, "objects": [], "people": 0}


## 2. Environment versions

**Result:** Python 3.13.16; torch 2.11.0+cu130 (CUDA 13.0); ultralytics 8.4.216; av 19.0.2; pyarrow 23.0.1; numpy 2.1.3; opencv 4.14.0.94; 2 × Tesla T4, driver 580.178.04.

In [10]:
%cd /kaggle/working/alethia-assignment
!git pull -q && git log --oneline -1
!mkdir -p /tmp/runs
!python --version
!pip list --format=freeze 2>/dev/null | grep -iE "^(ultralytics|torch|torchvision|av|pyarrow|numpy|opencv[a-z_-]*)=="
!python -c "import torch; print('torch.version.cuda =', torch.version.cuda)"
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader

/kaggle/working/alethia-assignment
b8c599b (HEAD -> main, origin/main, origin/HEAD) check.py compare --clip: score only reference seconds
Python 3.13.16
av==19.0.2
numpy==2.1.3
opencv-contrib-python==4.14.0.94
opencv-python==4.14.0.94
opencv-python-headless==4.14.0.94
pyarrow==23.0.1
torch==2.11.0+cu130
torchvision==0.26.0+cu130
ultralytics==8.4.216
torch.version.cuda = 13.0
Tesla T4, 580.178.04
Tesla T4, 580.178.04


## 3. Background helper `bg()`, then the FP32 / FP16 sample runs

This kernel refuses `!cmd &`, so `bg()` starts long jobs with `subprocess.Popen` and returns immediately. Output goes to a log file.

**What the sample runs are:**
- **Reference:** FP32, *every* frame, on the first 40 s of the first 32 videos of `train-00000` (1,270 seconds of video). It is built the same way as the grader's reference.
- **FP16 run:** identical except for precision.
- Both write per-frame detection counts (`--dump-frames`), so section 9 can simulate any frames-per-second setting offline, with no extra GPU time.

**Result:** FP32 took 381.6 s (about 50 fps per GPU) and FP16 took 162.0 s (about 114 fps per GPU), both exit 0.

In [13]:
import subprocess
def bg(cmd, log):
    """Run a shell command in the background; output goes to `log`. Returns immediately."""
    p = subprocess.Popen(["bash", "-c", cmd], stdout=open(log, "w"), stderr=subprocess.STDOUT,
                         start_new_session=True, cwd="/kaggle/working/alethia-assignment")
    print(f"started pid {p.pid}, log {log}")
    return p

samples = bg(
    "python run.py --input /tmp/fv/data --output /tmp/runs/ref.jsonl  --max-videos 32 --max-seconds 40 --no-half --dump-frames /tmp/runs/ref  > /tmp/runs/ref.log 2>&1; "
    "python run.py --input /tmp/fv/data --output /tmp/runs/fp16.jsonl --max-videos 32 --max-seconds 40 --half    --dump-frames /tmp/runs/fp16 > /tmp/runs/fp16.log 2>&1; "
    "echo ALL_DONE",
    "/tmp/runs/samples.log")

started pid 1192, log /tmp/runs/samples.log


## 4. Rotation check: 64 videos against OpenCV's autorotate

**Result:** `checked 64, rotated 0, mismatches 0`. Decoding matches OpenCV, but none of these videos has a rotation tag, so the rotation code itself is untested on real data.

In [12]:
!python check.py rotation --input /tmp/fv/data --max-videos 64 2>&1 | tail -25

checked 64, rotated 0, mismatches 0


## 5. Sample runs: quick progress check

In [17]:
!tail -2 /tmp/runs/ref.log; echo ---; tail -2 /tmp/runs/fp16.log 2>/dev/null; ls -la /tmp/runs | grep -c jsonl

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
---
4


## 6. Python 3.11 requirements test, first attempt (background)

Creates a clean Python 3.11 environment, installs `requirements.txt` with plain `pip` (as the grader does), and checks that torch is a CUDA 12.x build.

**Result: failed.** `av==19.0.2` needs Python ≥ 3.12, so `requirements.txt` was changed to `av==18.1.1`.

In [16]:
!git pull -q && git log --oneline -1 && cat requirements.txt
!pip install -q uv
reqtest = bg(
    "set -e; rm -rf /tmp/venv311; uv venv -q --seed -p 3.11 /tmp/venv311; "
    "/tmp/venv311/bin/python -m pip install -q -r requirements.txt; "
    "/tmp/venv311/bin/python -c \"import sys, torch, torchvision, ultralytics, av, pyarrow, cv2, numpy; "
    "print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', torch.cuda.is_available(), "
    "'ultralytics', ultralytics.__version__, 'av', av.__version__, 'cv2', cv2.__version__); "
    "assert sys.version_info[:2] == (3, 11); assert (torch.version.cuda or '').startswith('12.'), torch.version.cuda\"; "
    "echo REQ_TEST_PASS",
    "/tmp/runs/reqtest.log")

910eb24 (HEAD -> main, origin/main, origin/HEAD) Add requirements.txt (Python 3.11, torch 2.8.0 CUDA 12.8 build)
# Python 3.11. torch 2.8.0 / torchvision 0.23.0 from PyPI are CUDA 12.8 builds (no extra index needed).
torch==2.8.0
torchvision==0.23.0
ultralytics==8.4.216
av==19.0.2
pyarrow==23.0.1
numpy==2.1.3
opencv-python==4.14.0.94
started pid 1386, log /tmp/runs/reqtest.log


## 7. Python 3.11 requirements test, retry with `av==18.1.1`

**Expected:** `3.11.x torch 2.8.0+cu128 cuda 12.8 … REQ_TEST_PASS`. Check the result in section 8 (`tail /tmp/runs/reqtest.log`).

In [20]:
%cd /kaggle/working/alethia-assignment
!git pull -q && git log --oneline -1 && grep "^av" requirements.txt
reqtest = bg(
    "set -e; rm -rf /tmp/venv311; uv venv -q --seed -p 3.11 /tmp/venv311; "
    "/tmp/venv311/bin/python -m pip install -q -r requirements.txt; "
    "/tmp/venv311/bin/python -c \"import sys, torch, torchvision, ultralytics, av, pyarrow, cv2, numpy; "
    "print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', torch.cuda.is_available(), "
    "'ultralytics', ultralytics.__version__, 'av', av.__version__, 'cv2', cv2.__version__); "
    "assert sys.version_info[:2] == (3, 11); assert (torch.version.cuda or '').startswith('12.'), torch.version.cuda\"; "
    "echo REQ_TEST_PASS",
    "/tmp/runs/reqtest.log")

/kaggle/working/alethia-assignment
bbc99cb (HEAD -> main, origin/main, origin/HEAD) Pin av 18.1.1 for Python 3.11; use quantize=16 instead of deprecated half
av==18.1.1  # 19.x needs Python>=3.12
started pid 1716, log /tmp/runs/reqtest.log


## 8. Status of all background jobs (sample runs and requirements test)

In [22]:
!echo "=== sample runs (section 3) ==="; cat /tmp/runs/samples.log
!echo "--- ref (FP32) ---"; tail -2 /tmp/runs/ref.log 2>/dev/null || echo "not started"
!echo "--- fp16 ---"; tail -2 /tmp/runs/fp16.log 2>/dev/null || echo "not started yet (starts after ref)"
!echo "=== requirements test (section 7) ==="; tail -5 /tmp/runs/reqtest.log
!echo "=== processes ==="; ps -eo pid,etime,args | grep -E "run.py|pip install" | grep -v grep || echo "none running"
print("sample runs:", "running" if samples.poll() is None else f"finished, exit {samples.returncode}")
print("requirements test:", "running" if reqtest.poll() is None else f"finished, exit {reqtest.returncode}")

=== samples (cell 2) ===
ALL_DONE
--- ref (FP32) ---
[16:47:37] [gpu0] videos 17  frames 19109  50.5 fps  379s
[16:47:38] done in 381.6s -> /tmp/runs/ref.jsonl
--- fp16 ---
[16:50:21] [gpu0] videos 16  frames 18109  113.9 fps  159s
[16:50:22] done in 162.0s -> /tmp/runs/fp16.jsonl
=== requirements test (cell 5) ===
=== processes ===
   1716       00:52 bash -c set -e; rm -rf /tmp/venv311; uv venv -q --seed -p 3.11 /tmp/venv311; /tmp/venv311/bin/python -m pip install -q -r requirements.txt; /tmp/venv311/bin/python -c "import sys, torch, torchvision, ultralytics, av, pyarrow, cv2, numpy; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', torch.cuda.is_available(), 'ultralytics', ultralytics.__version__, 'av', av.__version__, 'cv2', cv2.__version__); assert sys.version_info[:2] == (3, 11); assert (torch.version.cuda or '').startswith('12.'), torch.version.cuda"; echo REQ_TEST_PASS
   1732       00:52 /tmp/venv311/bin/python -m pip install -q -r re

## 9. Sweep: frames per second vs accuracy, FP32 and FP16

Simulates analysing only *k* frames per second (`all` = every frame), using the per-frame dumps from section 3. It scores each setting with the grader's metrics against the FP32 every-frame reference.
The thresholds are F1 ≥ 0.95, people exact ≥ 0.85, people within 1 ≥ 0.95.

**Result:**
- FP16 changes nothing measurable.
- k = 1 fails; k ≥ 2 passes.
- **Chosen: FP16, k = 5**, which gives F1 0.982, exact 0.954, within 1 0.998. That leaves a margin of about 0.03 on F1, the tightest metric, for unseen videos.

These became the defaults in `run.py` (commit `06716f9`).

In [23]:
!echo "=== FP32, sampled ===";  python check.py sweep --ref "/tmp/runs/ref.*.jsonl" --k 0 1 2 3 4 5 6 8 10
!echo "=== FP16, sampled ===";  python check.py sweep --ref "/tmp/runs/ref.*.jsonl" --cand "/tmp/runs/fp16.*.jsonl" --k 0 1 2 3 4 5 6 8 10
!grep "fps" /tmp/runs/ref.log | tail -2; grep "fps" /tmp/runs/fp16.log | tail -2

=== FP32, sampled ===
32 videos, 1270 seconds
   k       F1    exact  within1  result
 all   1.0000   1.0000   1.0000  PASS
   1   0.9369   0.8732   0.9717  FAIL
   2   0.9584   0.9063   0.9882  PASS
   3   0.9699   0.9425   0.9953  PASS
   4   0.9794   0.9425   0.9937  PASS
   5   0.9825   0.9535   0.9984  PASS
   6   0.9822   0.9551   0.9976  PASS
   8   0.9893   0.9661   0.9992  PASS
  10   0.9899   0.9740   0.9992  PASS
=== FP16, sampled ===
32 videos, 1270 seconds
   k       F1    exact  within1  result
 all   1.0000   0.9984   1.0000  PASS
   1   0.9369   0.8732   0.9717  FAIL
   2   0.9584   0.9071   0.9882  PASS
   3   0.9699   0.9425   0.9953  PASS
   4   0.9794   0.9417   0.9937  PASS
   5   0.9819   0.9543   0.9984  PASS
   6   0.9822   0.9583   0.9976  PASS
   8   0.9899   0.9646   0.9992  PASS
  10   0.9910   0.9740   0.9992  PASS
[16:47:16] [gpu0] videos 16  frames 18005  50.5 fps  357s
[16:47:37] [gpu0] videos 17  frames 19109  50.5 fps  379s
[16:50:19] [gpu0] videos 15 

## 10. Initial timed run on files 0–1

It uses the grader's command with **only `--input` and `--output`**, so the defaults (FP16, 5 frames per second) are what gets measured. It runs from `/kaggle/working`, and bash's `time` adds the wall-clock time.

In [27]:
final = bg("cd /kaggle/working && time python /kaggle/working/alethia-assignment/run.py "
           "--input /tmp/fv/data --output /kaggle/working/results.jsonl; echo FINAL_DONE",
           "/tmp/runs/final.log")

started pid 1807, log /tmp/runs/final.log


## 11. Initial run: log head/tail and status

In [33]:
!head -3 /tmp/runs/final.log; echo ...; tail -2 /tmp/runs/final.log
print("final:", "running" if final.poll() is None else f"finished, exit {final.returncode}")

[17:04:39] 2 row groups, 66 videos, 2 worker(s) x 2 decoders, samples/sec=5, half=True
[17:04:55] [gpu1] videos 1  frames 466  35.4 fps  13s
[17:04:59] [gpu1] videos 2  frames 831  48.8 fps  17s
...
sys	0m55.291s
FINAL_DONE
final: finished, exit 0


## 12. Initial run: progress and ETA

Counts the videos finished so far from the partial output files the GPU workers write. The ETA is rough because videos vary in length.

In [32]:
import glob, json, re, time, os
log = open("/tmp/runs/final.log").read()
total = int(re.search(r"(\d+) videos", log).group(1))
start = time.mktime(time.strptime(time.strftime("%Y-%m-%d ") + re.search(r"\[(\d\d:\d\d:\d\d)\]", log).group(1), "%Y-%m-%d %H:%M:%S"))
vids, secs, errs = set(), 0, 0
for p in glob.glob("/kaggle/working/results.jsonl.part*"):
    for l in open(p):
        d = json.loads(l); vids.add(d["video"])
        if "error" in d: errs += 1
        else: secs += 1
el = time.time() - start
done = len(vids)
print(f"videos {done}/{total}  ({done/total:.0%})   video time done {secs/3600:.2f} h   errors {errs}")
print(f"elapsed {el/60:.1f} min" + (f"   ETA ~{el/done*(total-done)/60:.0f} min (rough: videos vary in length)" if done else ""))
print("status:", "running" if final.poll() is None else f"finished, exit {final.returncode}")
!grep fps /tmp/runs/final.log | tail -2

videos 0/66  (0%)   video time done 0.00 h   errors 0
elapsed 19.0 min
status: finished, exit 0
[17:11:56] [gpu1] videos 34  frames 46999  108.2 fps  434s
[17:12:05] [gpu0] videos 32  frames 46894  105.7 fps  444s


## 13. Initial run: summary

**Result:**
- `done in 447.9s`: 66 videos, 5.22 video-hours, 0 errors, 18,806 lines.
- That is about **42× real time** on 2 × T4, with each GPU analysing about 106 frames per second.

*(This cell's code was corrected after the run; rerun it in the same session to refresh the output.)*

In [ ]:
!grep -E "row groups|done in|^real|^user|^sys" /tmp/runs/final.log
!grep "fps" /tmp/runs/final.log | tail -2
import json, collections
n, vids, errs, secs = 0, set(), [], collections.Counter()
for l in open("/kaggle/working/results.jsonl"):
    d = json.loads(l); vids.add(d["video"]); n += 1
    if "error" in d: errs.append(d)
    else: secs[d["video"]] += 1
print(f"lines {n}, videos {len(vids)}, video-hours {sum(secs.values())/3600:.2f}, errors {len(errs)}")
for e in errs: print("  ", e)


## 14. Initial run vs reference (clipped compare)

Scores the real run's output on just the reference's seconds (the first 40 s of 32 videos).

**Result:** PASS, with F1 0.9819, exact 0.9543 and within 1 0.9984. That is identical to the simulated FP16, k = 5 row, so the sweep reproduces the real pipeline exactly.
Per class: laptop 0.915 and cell phone 0.939 are below 0.95 on their own, with only about 30 positives each. The pooled F1 passes, and so does the class average (0.960).

In [35]:
!python /kaggle/working/alethia-assignment/check.py compare /tmp/runs/ref.jsonl /kaggle/working/results.jsonl --clip

errors: ref 0, cand 0
videos 32  seconds 1270  missing seconds in candidate 0
object F1 0.9819  people exact 0.9543  within1 0.9984  -> PASS
   person      F1 0.9876  (tp 716, fp 6, fn 12)
   car         F1 0.9565  (tp 33, fp 1, fn 2)
   dog         F1 1.0000  (tp 7, fp 0, fn 0)
   laptop      F1 0.9153  (tp 27, fp 1, fn 4)
   cell phone  F1 0.9394  (tp 31, fp 2, fn 2)


# Batch run: files 2 onward

Resumable runner (`kaggle_batches.py`), 3 parquet files per batch: 02 (files 2–4), 05, 08, … 47.
At the measured 42× real time, a batch (about 7.5 h of video) should take about 11 min, plus about 1 min to download.

Run **A**, watch with **B**, stop with **C** (or `CUTOFF` in A), then merge with **D**.

## A. Launch the batch runner

Pulls the latest code, deletes the old Python 3.11 test environment to free disk, adds files 0–1 to `manifest.jsonl`, then works through the batches.
For each batch it downloads to `/tmp`, runs `run.py` with defaults, keeps `res_NN.jsonl` only if `run.py` exits 0, and deletes the batch's data. The HF token is read from the Kaggle secret and never printed.

- **Good:** commit `da90c13` or newer, plenty of free space, `ls` shows **only** train-00000 and train-00001, then `runner pid …`.
- If `/tmp/fv/data` lists more than files 0 and 1, **stop and check before launching**. The extra files would be reported missing at merge.
- **To resume after a stop:** run this cell again. Finished batches are skipped, and files 0–1 aren't added twice.
- `CUTOFF` is an optional hard stop enforced by the runner itself, so it still works if the notebook kernel restarts. Use it *or* cell C, not both.

In [ ]:
import os, subprocess, signal
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # inherited by the runner; never printed
%cd /kaggle/working/alethia-assignment
!git pull -q && git log --oneline -1
!rm -rf /tmp/venv311 && df -h /tmp | tail -1
!ls /tmp/fv/data    # must list ONLY train-00000 and train-00001 (the files behind results.jsonl)

CUTOFF = ""   # optional hard cutoff, "HH:MM" on Kaggle's clock (UTC; check with !date)
cmd = ["python", "kaggle_batches.py", "--seed-manifest", "/tmp/fv/data"] + (["--stop-at", CUTOFF] if CUTOFF else [])
batches = subprocess.Popen(cmd, stdout=open("/kaggle/working/batches.out", "a"), stderr=subprocess.STDOUT,
                           start_new_session=True, cwd="/kaggle/working/alethia-assignment")
print("runner pid", batches.pid)

## B. Watch progress

Rerun whenever you want an update.

- **Good:** lines like `batch 02 files 2-4 (...): OK videos 98 video_hours 7.40 errors 0 elapsed 690s (39x realtime) download 55s`, a growing list of `res_NN.jsonl`, and roughly constant free space on `/tmp`.
- **If a line says `FAILED` or `download failed`:** look at `!tail -20 /kaggle/working/batch_logs/run_NN.log`. The runner moves on by itself.

In [ ]:
!echo "=== batches.log ==="; tail -8 /kaggle/working/batches.log
!echo "=== current batch ==="; f=$(ls -t /kaggle/working/batch_logs/run_*.log 2>/dev/null | head -1); [ -n "$f" ] && echo "$f" && head -1 "$f" && grep fps "$f" | tail -2
!echo "=== done ==="; ls -1 /kaggle/working/res_*.jsonl 2>/dev/null; df -h /tmp | tail -1
print("runner:", "running" if batches.poll() is None else f"stopped, exit {batches.returncode}")

## C. Stop at a set time, or right now

`STOP_AT` is on Kaggle's clock (UTC; check with `!date`). The value in the cell is only an example. Set `""` to stop immediately.

- **Good:** `will stop at …` or `stop signal sent …`. Afterwards, cell B shows `stopped by signal 15; current batch discarded` (or `cutoff …`) and `runner: stopped`.
- Check that no partial file is left: `!ls /kaggle/working/*.tmp` should find nothing.

In [ ]:
import threading, datetime, signal, subprocess
STOP_AT = "18:20"   # EXAMPLE - set your cutoff, Kaggle clock (UTC). "" = stop now.

def stop_runner():
    try:
        batches.send_signal(signal.SIGTERM)          # runner kills run.py and discards the partial batch
    except NameError:                                 # kernel restarted and lost `batches`
        subprocess.run(["pkill", "-TERM", "-f", "kaggle_batches.py"])
    print("stop signal sent at", datetime.datetime.now().strftime("%H:%M:%S"))

if STOP_AT:
    now = datetime.datetime.now()
    t = now.replace(hour=int(STOP_AT[:2]), minute=int(STOP_AT[3:]), second=0, microsecond=0)
    threading.Timer(max(0, (t - now).total_seconds()), stop_runner).start()
    print(f"will stop at {STOP_AT} (in {(t - now).total_seconds() / 60:.0f} min)")
else:
    stop_runner()

## D. Merge, check, gzip

**Run this only after the runner has stopped** (cell B shows `runner: stopped`). With "Run all", it would run while batches are still going, and the merged file would be incomplete.

It merges `results.jsonl` and every complete `res_NN.jsonl`, never the `.tmp` files. Then it checks for:
- duplicate videos
- missing videos, against `manifest.jsonl`
- gaps in any video's seconds

If a check fails, it lists the problems and writes nothing. Otherwise it compresses the result with gzip, verifies the archive, and compares line counts.

- **Good:** `files covered: N (indices 0-…)`, the video count, video-hours, `checks: … OK`, `gzip OK`, and equal raw and gzipped line counts.
- This output fills the coverage tables in the README and HARDWARE.md.

In [ ]:
%cd /kaggle/working/alethia-assignment
!python merge_results.py --dir /kaggle/working --out /kaggle/working/results_all.jsonl
!cd /kaggle/working && gzip -9 -kf results_all.jsonl && gzip -t results_all.jsonl.gz && echo "gzip OK" \
  && echo "lines: $(wc -l < results_all.jsonl) raw, $(zcat results_all.jsonl.gz | wc -l) gz" && ls -lh results_all.jsonl*
!cat /kaggle/working/batches.log